# Entraînement LSTM (Google Colab)

Ce notebook entraîne le modèle symbolique (ABC → MIDI) sur GPU Colab.

**À la fin**, téléchargez `best_model.pth` et `vocab.json`, puis placez-les à la racine du projet local.

Le dataset Hugging Face se télécharge automatiquement — aucun upload manuel.

Runtime recommandé : **GPU** (Runtime → Change runtime type → T4).

In [ ]:
!pip install -q torch datasets tqdm matplotlib

import json
import os
import random
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Données

In [ ]:
def extract_abc_notation(text):
    if "Assistant:" in text:
        text = text.split("Assistant:", 1)[1].strip()
    return text.replace("</s>", "").strip()

print("Loading dataset...")
dataset = load_dataset("Seeker38/music_abc_notation", split="train")
abc_songs = [extract_abc_notation(item["output"]) for item in dataset]
abc_songs = [s for s in abc_songs if 50 < len(s) < 2000]
print(f"Songs ready: {len(abc_songs)}")

In [ ]:
all_text = "\n".join(abc_songs)
vocab = sorted(set(all_text))
vocab_size = len(vocab)
char2idx = {c: i for i, c in enumerate(vocab)}
idx2char = {i: c for i, c in enumerate(vocab)}

vocab_info = {
    "vocab": vocab,
    "char2idx": char2idx,
    "idx2char": {int(k): v for k, v in idx2char.items()},
    "vocab_size": vocab_size,
}
with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_info, f, ensure_ascii=False, indent=2)

print(f"Vocabulary: {vocab_size} characters")

In [ ]:
class MusicDataset(Dataset):
    def __init__(self, songs, char2idx, seq_length=64):
        self.char2idx = char2idx
        self.seq_length = seq_length
        self.sequences = []
        for song in songs:
            if len(song) > seq_length:
                step = seq_length // 2
                for i in range(0, len(song) - seq_length, step):
                    seq = song[i : i + seq_length + 1]
                    if len(seq) == seq_length + 1:
                        self.sequences.append(seq)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        x = [self.char2idx[c] for c in seq[:-1]]
        y = [self.char2idx[c] for c in seq[1:]]
        return torch.tensor(x), torch.tensor(y)

SEQ_LENGTH = 64
BATCH_SIZE = 256  # GPU Colab
DATASET_FRACTION = 0.4

random.seed(42)
sampled = random.sample(abc_songs, int(len(abc_songs) * DATASET_FRACTION))
split = int(0.95 * len(sampled))
train_ds = MusicDataset(sampled[:split], char2idx, SEQ_LENGTH)
val_ds = MusicDataset(sampled[split:], char2idx, SEQ_LENGTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=True, num_workers=2)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Batch: {BATCH_SIZE}")

## 2. Modèle

In [ ]:
class MusicLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            embedding_dim, hidden_dim, num_layers, dropout=dropout, batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, hidden=None):
        embedded = self.embedding(x)
        if hidden is None:
            out, hidden = self.lstm(embedded)
        else:
            out, hidden = self.lstm(embedded, hidden)
        out = self.dropout(out)
        return self.fc(out), hidden

model = MusicLSTM(vocab_size=vocab_size).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Entraînement

In [ ]:
EPOCHS = 12
EARLY_STOPPING_PATIENCE = 3

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)
history = {"train_loss": [], "val_loss": [], "learning_rates": []}


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total = 0.0
    for inputs, targets in tqdm(loader, desc="train", leave=False):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs, _ = model(inputs)
        loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total = 0.0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs, _ = model(inputs)
        loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
        total += loss.item()
    return total / len(loader)


best_val = float("inf")
patience = 0

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    val_loss = validate(model, val_loader, criterion)
    scheduler.step(val_loss)
    lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["learning_rates"].append(lr)

    print(f"  train={train_loss:.4f}  val={val_loss:.4f}  lr={lr:.6f}")

    if val_loss < best_val:
        best_val = val_loss
        patience = 0
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "vocab_size": vocab_size,
            },
            "best_model.pth",
        )
        print("  best_model.pth saved")
    else:
        patience += 1
        print(f"  no improvement ({patience}/{EARLY_STOPPING_PATIENCE})")
        if patience >= EARLY_STOPPING_PATIENCE:
            print("Early stopping.")
            break

print("Training done.")

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.legend()
plt.title("Loss")
plt.subplot(1, 2, 2)
plt.plot(history["learning_rates"])
plt.title("Learning rate")
plt.tight_layout()
plt.savefig("training_history.png", dpi=120)
plt.show()

## 4. Téléchargement

Après téléchargement, copiez `best_model.pth` et `vocab.json` à la racine du dépôt local, puis lancez `python app.py` (onglet MIDI).

In [ ]:
zip_name = "lstm_weights.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in ("best_model.pth", "vocab.json", "training_history.png"):
        if os.path.exists(name):
            zf.write(name)
            print(f"packed: {name}")

try:
    from google.colab import files

    files.download(zip_name)
    print(f"Download started: {zip_name}")
except ImportError:
    print(f"Not on Colab — zip created locally: {zip_name}")